# 📊 Conteo del Dataset — ExpoEscom
Cuenta las imágenes de cada clase dentro de una carpeta de Drive,
**incluyendo carpetas compartidas** ("Compartido conmigo").
Solo lectura: no descarga ni modifica nada.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
service = build('drive', 'v3')
print('✅ Autenticado con Drive API')

In [ ]:
# ════ ÚNICA CONFIGURACIÓN NECESARIA ══════════════════════════

FOLDER_ID = '1p3-ro7-JFrX2p0kQ8wm6DO_nk1f6O_AF'

# ─────────────────────────────────────────────────────────────
# Meta de imágenes por clase (mismos valores que ToonVerse)
CLASES_META = {
    'pokemon'          : 100_000,
    'one_piece'        : 100_000,
    'dragon_ball'      : 100_000,
    'bleach'           : 100_000,
    'ben_10'           : 100_000,
    'hora_de_aventura' : 100_000,
    'un_show_mas'      : 100_000,
    'naruto'           : 100_000,
    'doraemon'         : 100_000,
    'yu_gi_oh'         : 100_000,
    'barbie'           : 100_000,
    'attack_on_titan'  : 100_000,
    'simpson'          : 100_000,
    'star_wars'        : 100_000,
    'otra'             : 1_200_000,
}

EXTENSIONES_IMAGEN = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif'}
print(f'Folder ID : {FOLDER_ID}')
print(f'Clases    : {len(CLASES_META)}')

In [ ]:
# ════ HELPERS DE DRIVE API ════════════════════════════════════
from pathlib import Path

_ARGS = dict(supportsAllDrives=True, includeItemsFromAllDrives=True,
             pageSize=1000)
_FOLDER_MIME = 'application/vnd.google-apps.folder'


def listar_subcarpetas(parent_id):
    """Devuelve {nombre: id} de las subcarpetas directas de parent_id."""
    carpetas, token = {}, None
    while True:
        resp = service.files().list(
            q=f"'{parent_id}' in parents and mimeType='{_FOLDER_MIME}' "
              f"and trashed=false",
            fields='nextPageToken, files(id,name)',
            pageToken=token, **_ARGS).execute()
        for f in resp.get('files', []):
            carpetas[f['name']] = f['id']
        token = resp.get('nextPageToken')
        if not token:
            break
    return carpetas


def contar_imagenes(folder_id):
    """Cuenta recursivamente los archivos-imagen bajo folder_id."""
    total, stack = 0, [folder_id]
    while stack:
        fid, token = stack.pop(), None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType)',
                pageToken=token, **_ARGS).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == _FOLDER_MIME:
                    stack.append(f['id'])
                elif (f['mimeType'].startswith('image/') or
                      Path(f['name']).suffix.lower() in EXTENSIONES_IMAGEN):
                    total += 1
            token = resp.get('nextPageToken')
            if not token:
                break
    return total


print('✅ Helpers listos')

In [ ]:
# ════ CONTEO ══════════════════════════════════════════════════

raiz = service.files().get(
    fileId=FOLDER_ID, fields='name', supportsAllDrives=True).execute()
print(f'Carpeta raíz : {raiz["name"]}')

subcarpetas = listar_subcarpetas(FOLDER_ID)
print(f'Subcarpetas  : {len(subcarpetas)} encontradas\n')

print('═' * 62)
print(f'  {"CLASE":<22}  {"TIENE":>10}  {"META":>10}  {"FALTA":>10}')
print('═' * 62)

total_tiene = total_meta = clases_listas = 0

for clase, meta in CLASES_META.items():
    total_meta += meta
    if clase not in subcarpetas:
        print(f'  ⬜ {clase:<20}  {"—":>10}  {meta:>10,}  {"no existe":>10}')
        continue

    print(f'  ⏳ {clase:<20}  contando…', end='\r', flush=True)
    n = contar_imagenes(subcarpetas[clase])
    falta  = max(0, meta - n)
    estado = '✅' if n >= meta else ('🔄' if n > 0 else '⬜')
    print(f'  {estado} {clase:<20}  {n:>10,}  {meta:>10,}  {falta:>10,}')

    total_tiene += n
    if n >= meta:
        clases_listas += 1

print('═' * 62)
print(f'  {"TOTAL":<22}  {total_tiene:>10,}  {total_meta:>10,}  '
      f'{max(0, total_meta - total_tiene):>10,}')
print('═' * 62)
print(f'\nClases completas : {clases_listas} / {len(CLASES_META)}')
print(f'Progreso total   : {100 * total_tiene / total_meta:.1f}%')

# ── Subcarpetas que existen pero NO están en CLASES_META ──────
extra = [n for n in subcarpetas if n not in CLASES_META]
if extra:
    print(f'\nℹ️  Subcarpetas no esperadas: {extra}')